# BTW Module 02: Differential Expression (DE) Integration
**Downstream Bulk Transcriptomics Workbench (`btw`)**

สมุดงานตัวอย่างสาธิตการใช้งาน **FR-3 (Differential Expression Integration)**:
1. การสร้างและฟิตโมเดล PyDESeq2 ผ่าน Thin Helper (`build_deseq_dataset`)
2. การวิเคราะห์ยีนแสดงออกแตกต่างและดึงตารางผลลัพธ์มาตรฐาน (`run_deseq_stats` -> `DEResult`)
3. การสืบค้นยีนกลุ่ม UP / DOWN / DEGs และการสรุปผลการวิเคราะห์
4. การรัน Multiple Contrasts พร้อมกันเป็นชุด (`run_multiple_contrasts` -> `MultiContrastResult`)
5. การเปรียบเทียบวิธีการปรับแก้ Multiple Testing Correction ต่าง ๆ ผ่าน `statsmodels`
6. การส่งออกตารางผลลัพธ์เป็นสมุดงาน Excel หลายชีต

In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd

import btw
from btw import set_seed, logger
from btw.de_analysis import (
    build_deseq_dataset,
    run_deseq_stats,
    run_de,
    run_multiple_contrasts,
    apply_multitest_correction,
    SUPPORTED_METHODS,
)

# กำหนด seed กลางเพื่อความเที่ยงตรง
set_seed(42)
print(f"BTW version: {btw.__version__}")

## 1. เตรียมชุดข้อมูลตัวอย่าง (Synthetic Bulk Counts & Metadata)
สร้าง Count matrix (100 genes x 6 samples) ประกอบด้วยกลุ่ม control (3 ซ้ำ) และ treated (3 ซ้ำ) พร้อมปัจจัย batch

In [ ]:
genes = [f"GENE_{i:03d}" for i in range(1, 101)]
samples = ["ctrl_1", "ctrl_2", "ctrl_3", "treat_1", "treat_2", "treat_3"]

np.random.seed(42)
base_counts = np.random.negative_binomial(5, 0.01, size=(100, 6))
# Injected differential expression: ยีน 1-15 up-regulated, 16-30 down-regulated ในกลุ่ม treated
base_counts[:15, 3:] = (base_counts[:15, 3:] * 4.5).astype(int)
base_counts[15:30, 3:] = (base_counts[15:30, 3:] * 0.2).astype(int)
counts_df = pd.DataFrame(np.clip(base_counts, 0, None), index=genes, columns=samples)

metadata_df = pd.DataFrame({
    "sample_id": samples,
    "condition": ["control", "control", "control", "treated", "treated", "treated"],
    "batch": ["batch1", "batch2", "batch1", "batch2", "batch1", "batch2"],
}).set_index("sample_id")

display(metadata_df)

## 2. การสร้างและฟิตโมเดล PyDESeq2 (Thin Helper: FR-3)
ฟังก์ชัน `build_deseq_dataset` จะทำการตรวจสอบความสอดคล้องของข้อมูล, สลับแถวคอลัมน์ให้อยู่ในรูปแบบที่ PyDESeq2 ต้องการ (samples x genes) โดยอัตโนมัติ และฟิตโมเดล Negative Binomial GLM

In [ ]:
dds = build_deseq_dataset(
    counts=counts_df,
    metadata=metadata_df,
    design_factors="condition",
    ref_level={"condition": "control"},
    fit_model=True,
)
print(dds)

## 3. การรันสถิติและวิเคราะห์ผล DE (Single Contrast: FR-3)
รันเปรียบเทียบ `condition: treated vs control` ผ่าน `run_deseq_stats`

In [ ]:
de_result = run_deseq_stats(
    dds=dds,
    contrast=("condition", "treated", "control"),
    alpha=0.05,
    lfc_threshold=1.0,
)

# แสดงสรุปผลสถิติ
print(de_result.summary())

# แสดงตารางผลลัพธ์มาตรฐาน
display(de_result.results_df.head(10))

### การสืบค้นยีนกลุ่มสำคัญ (DEGs, UP, DOWN)

In [ ]:
up_genes = de_result.get_up_genes()
down_genes = de_result.get_down_genes()
print(f"จำนวนยีน Up-regulated ({len(up_genes)} ยีน): {up_genes[:5]} ...")
print(f"จำนวนยีน Down-regulated ({len(down_genes)} ยีน): {down_genes[:5]} ...")

# ตาราง DEGs ที่มีนัยสำคัญทางสถิติ
sig_degs = de_result.get_degs()
print(f"\nตารางยีนที่มีนัยสำคัญทั้งหมด ({sig_degs.shape[0]} ยีน):")
display(sig_degs.head(10))

## 4. การรัน Multiple Contrasts ในรอบเดียว (Multi-Contrast Batch Execution)
สามารถกำหนดหลายการเปรียบเทียบในรูปแบบ List of tuples แล้วรันใน loop เดียวกัน พร้อมรวมผลเป็น Master Table

In [ ]:
contrasts_list = [
    ("condition", "treated", "control"),
    ("batch", "batch2", "batch1"),
]

multi_result = run_multiple_contrasts(
    counts=counts_df,
    metadata=metadata_df,
    contrasts=contrasts_list,
    alpha=0.05,
    lfc_threshold=1.0,
)

# ตารางสรุปภาพรวมข้าม Contrasts
display(multi_result.summary_table())

# Master Table รวมผลลัพธ์
print("\nMaster Table:")
display(multi_result.master_table.head(10))

## 5. การเปรียบเทียบวิธีการปรับแก้ Multiple Testing Correction (statsmodels Integration)
ทดสอบเปรียบเทียบผลระหว่าง Benjamini-Hochberg (`fdr_bh`) กับ Bonferroni (`bonferroni`)

In [ ]:
df_raw = de_result.results_df[["pvalue", "log2FoldChange"]].copy()

# 1. ปรับแก้ด้วย Bonferroni (เข้มงวดสูง)
df_bonf = apply_multitest_correction(
    df_raw, pvalue_col="pvalue", method="bonferroni", alpha=0.05,
    output_padj_col="padj_bonferroni", output_sig_col="sig_bonferroni"
)

# 2. ปรับแก้ด้วย Benjamini-Hochberg (มาตรฐาน FDR)
df_bh = apply_multitest_correction(
    df_raw, pvalue_col="pvalue", method="fdr_bh", alpha=0.05,
    output_padj_col="padj_bh", output_sig_col="sig_bh"
)

comparison_df = pd.DataFrame({
    "raw_pvalue": df_raw["pvalue"],
    "padj_BH": df_bh["padj_bh"],
    "sig_BH": df_bh["sig_bh"],
    "padj_Bonferroni": df_bonf["padj_bonferroni"],
    "sig_Bonferroni": df_bonf["sig_bonferroni"],
})
display(comparison_df.head(10))
print(f"Significant genes with BH: {df_bh['sig_bh'].sum()}")
print(f"Significant genes with Bonferroni: {df_bonf['sig_bonferroni'].sum()}")

## 6. การส่งออกผลการวิเคราะห์ DE (Export to Multi-Sheet Excel)
ส่งออกตารางสรุป, Master Table, และผล DE ของแต่ละ contrast ลงในสมุดงาน Excel เล่มเดียว

In [ ]:
out_dir = Path("results/example_de")
out_dir.mkdir(parents=True, exist_ok=True)

report_file = out_dir / "differential_expression_report.xlsx"
saved_path = multi_result.export_excel(report_file)
print(f"Successfully saved multi-contrast DE report to: {saved_path}")